# Two-qutrit polytope generation

For dimensions higher that two qubits, the inequalities defined in Phys. Rev. A 109, 012423 (2024) no longer completely determine the region of valid quantum states in terms of the Bloch lengts. The inequalities remain valid but not tight, and additional constraints are required, since points inside the derived region do not correspond to valid states.

In this notebook we explore the optimization of the Bloch lengths among the set of points determined by the inequalities in order to numerically find the valied region. This region will later be used for optimization of the entanglement. This is a previous step required for the generation of figures 2 (a), 1 (b), 2 (d) and 2 (e) of the paper.

# Importations

In [ ]:
# Numerical and scientific python programming
import numpy as np

import matplotlib.pyplot as plt

# Auxiliary python functions
import time

# Local importations
from moments.bloch import (generate_gell_mann_basis, compute_tensor_basis, compute_subset_index_map,
                           compute_bipartite_region_upper, compute_bipartite_region_lower)
from moments.initialization import compute_initial_param_repeat

# Saving
from moments.saving import find_project_root

# Hilbert apace definition

First we define some variables that will be used through out most of the functions.

For an $N$ qubit system with indices taking values $n = 1, \ldots, N$, we now change the convention to follow python's indexing $n \mapsto n - 1$, such that $N = 0, \ldots, N - 1$.

- The dimenstion of each local Hilbert space is indicated by the variable `dim`: each entry `dim[n]` corresponds to $d_n$.

Variables `basis`, `local_bases` and `local_basis_sizes` are intermidiate steps to compute `tensor_basis` and `subset_index_map`.

- The variabe `tensor_basis` is a numpy array containing the basis $\{ \mu_i \}_{i = 0}^{d^2 - 1}$ of the Hilbert space $\mathbb H$. Since $\mathcal H = \bigotimes_{n \in \mathbf N} \mathcal H_{n-1}$, then erach basis element is expanded as $\mu_{i_1}^1 \otimes \cdots \otimes \mu_{i_N}^N$. These are ordered in lexycographic order.

- The variable `subset_index_map` is a dictionary that as keys has every possible subset $\mathbf M \subseteq \mathbf N$ of the set of sub-systems. The value of each key corresponds to the indices $i$ of the bloch vector $r_i$ that describe the subsystem $\mathbf M$, according to equation (3) of the paper. This serves to indicate several routines which elements to use if only some subsystems are to be taken into account.

The space of quantum states in terms of the Bloch lengts is a subset of $\mathbb R^3$, where each coordinate corresponds to one Bloch length and they satisfy several conditions. For more information we refer to Phys. Rev. A 109, 012423 (2024) or section 3.3 of the paper.

Denote $|\vec r_1| := x$, $|\vec r_2| := y$ and $|\vec r_{12}| := z$. The particular conditions for a two-qubit system read:

- $(x, y, z) \in [0, \sqrt2] \times [0, \sqrt2] \times [0, 2\sqrt2]$,
- $z \ge \sqrt2 (x + y) - 2$,
- $z^2 \le 8 + 2 (x^2 + y^2) - 6xy - 3 \sqrt6 |x - y|$.

In [ ]:
# Define system parameters.
dn = 3
dim = [dn, dn]
N, d = len(dim), int(np.prod(dim))

# Initialize the Pauli basis of a one-qubit system.
basis = generate_gell_mann_basis(dn)
local_bases = [basis.copy()] * N
local_basis_sizes = [len(lb) for lb in local_bases]

# Compute tensor basis of the N qubit system.
tensor_basis = compute_tensor_basis(local_bases)
# Compute index mappings from basis elements to Bloch vector elements.
subset_index_map = compute_subset_index_map(local_basis_sizes)

# Symetric plane

We first focuss in the the intersection of this region in the first quadrant and the plane $x=y$. This simplifies the conditions to:

- $(x, z) \in [0, \sqrt2] \times [0, 2\sqrt2]$,
- $z \ge 2 (\sqrt2 x - 1)$,
- $z \le \sqrt{8 - 2 x^2}$.

For easier visualization, we can call $t$ the distance to the $z$-axis from the points in the $x=y$ plane. Thus, change the coordinates to a 2D representation where we use the intersection between the $x=y$ and $z=0$ planes as first coordinate, and $z$ as the second coordinate. Then, we would have $t = \sqrt{x^2 + y^2} = \sqrt2 x$ or $x = y = t/\sqrt2$, which transforms the inequalities as:

- $(t, x) \in [0, 2] \times [0, 2\sqrt2]$,
- $z \ge 2(t - 1)$,
- $z \le \sqrt{8 - t^2}$.

In [ ]:
# Define the domain where the function will be evaluated.
t = np.linspace(0, 2, 1000)
x = t / np.sqrt(2)

# Evaluate the boundaries of the different regions of the domain.
z_upper = compute_bipartite_region_upper(dn, x, x)
z_lower = compute_bipartite_region_lower([dn, dn], x, x)

# Plot the regions indicating the separation between the different behavours.
fig, ax = plt.subplots(figsize=(6, 8))

ax.fill_between(t, z_upper, z_lower, alpha=0.4, color="blue")

ax.plot(t, z_upper, "red", label="Upper boundary")
ax.plot(t, z_lower, "blue", label="Lower boundary")

ax.set(xlabel = r"$t = \sqrt{2} x$", ylabel = r"$z$", title = "Intersection of region with $x=y$ plane",
       xlim = (0, 2.1), ylim = (0, 3))
ax.grid(True, alpha=0.5, linestyle="--")
ax.legend()
plt.show()

## Moment space discretization

To iterate over valid quantum states and perform the optimization, we create a grid parametrized by the phisically allowed Bloch lengths.

- The variable `D` is an integer indicating the number of points used ($D + 1$) for the primary grid coordinate ($t$). Then, this is rescaled for the rest of the coordinates (`Dt` and `Dz`) to get a homogenous cover of the space (in the $t, z$ representation).

- The variables `x`, and `z` are numpy arrays that discretize each coordinate. Afther that the two-dimensional coordinate mesh is defined with variables `X` and `Z`.

- The varibale `indices` extracts the indices in the grid for the loop.

In [ ]:
# Determine the number of grid points along each coordinate.
D = 20
Dt, Dz = D + 1, int(np.sqrt(2) * D + 1)

# Discretize each coordinate.
t = np.linspace(0, 2, Dt)
x = t / np.sqrt(2)
z = np.linspace(0, 2*np.sqrt(2), Dz)

# Construct the two-dimensional coordinate mesh.
X, Z = np.meshgrid(x, z, indexing='ij')

# Extract the indices of all grid points for the loop.
indices = np.ndindex(X.shape)
total_points = np.prod(X.shape)
print(f"Number of points: {total_points}")

## Space exploration

In this section, for each point in the grid, we find a quantum state with the closest Bloch lengths to the target ones. If the program is not able to find a quantum state, se lable the point outside the valid quantum state region. We quantify the degree of convergence by the distance from the Bloch lengths of the result with respect to the target ones.

- Results of the optimiztion are stored in the numpy array `bloch_distance`. We are also storing the time taken for each point to optimize in `times`.

In [ ]:
# List to store the time for each point.
times = []
# Store the results of the optimization.
sym_distance = np.empty_like(X, dtype=float)

# Iterate over every grid point.
for counter, (idx, jdx) in enumerate(indices, 1):
    t0 = time.time()

    # Read Bloch lengths for each point.
    x_val = X[idx, jdx]
    z_val = Z[idx, jdx]

    # Construct Boch length constraints.
    Rt = {(1,): float(x_val), (2,): float(x_val), (1, 2): float(z_val)}

    # Solve the optmization problem. Find the closest quantum state with the given moments.
    param_res = compute_initial_param_repeat(d, tensor_basis, subset_index_map, Rt)
    sym_distance[idx, jdx] = sum(list(param_res.checks["moments_distance"].values()))

    tf = time.time()
    times.append(tf-t0)
    
    # Display status of the loop.
    percent_complete = (counter / total_points) * 100
    print(f"\rProgress: {percent_complete:.2f}% ({counter}/{total_points}) | Time for this point: {tf-t0:.3f} s", end="", flush=True)

print(f"\n\nDone!")
print(f"Total time: {sum(times)/60:.2f} min | Average time per point: {sum(times)/total_points:.3f} s", )

# $xz$-plane

In the intersection of the region and the plane $y=0$, the inequalities become:

We now focuss in the the intersection of this region in the first quadrant and the plane $y=0$. This simplifies the conditions to:

- $(x, z) \in [0, \sqrt2] \times [0, 2\sqrt2]$,
- $z \ge \sqrt2 x - 2$,
- $z^2 \le 8 - 3 \sqrt6 x + 2 x^2$.

In [ ]:
# Define the domain where the function will be evaluated.
x = np.linspace(0, np.sqrt(2), 1000)

# Evaluate the boundaries of the different regions of the domain.
z_upper = compute_bipartite_region_upper(dn, x, 0)
z_lower = compute_bipartite_region_lower([dn, dn], x, 0)

# Plot the regions indicating the separation between the different behavours.
fig, ax = plt.subplots(figsize=(4.5, 8))

ax.fill_between(x, z_upper, z_lower, alpha=0.4, color='blue')

ax.plot(x, z_upper, "red", label=r"$z = \sqrt{8 - 3\sqrt{6}x + 2x^2}$")
ax.plot(x, z_lower, "blue", label=r"$z = \max(0, \sqrt{2} x - 2)$")

ax.set(xlabel = r"$x$", ylabel = r"$z$", title = "Intersection of region with $x=y$ plane",
       xlim = (0, 1.5), ylim = (0, 3))
ax.grid(True, alpha=0.5, linestyle="--")
ax.legend()
plt.show()

## Moment space discretization

## Moment space discretization

To iterate over valid quantum states and perform the optimization, we create a grid parametrized by the phisically allowed Bloch lengths.

- The variable `D` is an integer indicating the number of points used ($D + 1$) for the primary grid coordinate ($x$). Then, this is rescaled for the rest of the coordinates (`Dx` and `Dz`) to get a homogenous cover of the space (in the $x, z$ representation).

- The variables `x`, and `z` are numpy arrays that discretize each coordinate. Afther that the two-dimensional coordinate mesh is defined with variables `X` and `Z`.

- The varibale `indices` extracts the indices in the grid for the loop.

In [ ]:
# Determine the number of grid points along each coordinate.
D = 200
Dx, Dz = D + 1, 2 * D + 1

# Discretize each coordinate.
x = np.linspace(0, np.sqrt(2), Dx)
z = np.linspace(0, 2*np.sqrt(2), Dz)

# Construct the two-dimensional coordinate mesh.
X, Z = np.meshgrid(x, z, indexing='ij')

# Extract the indices of all grid points for the loop.
indices = np.ndindex(X.shape)
total_points = np.prod(X.shape)
print(f"Number of points: {total_points}")

## Space exploration

In this section, for each point in the grid, we find a quantum state with the closest Bloch lengths to the target ones. If the program is not able to find a quantum state, se lable the point outside the valid quantum state region. We quantify the degree of convergence by the distance from the Bloch lengths of the result with respect to the target ones.

- Results of the optimiztion are stored in the numpy array `bloch_distance`. We are also storing the time taken for each point to optimize in `times`.

In [ ]:
# List to store the time for each point.
times = []
# Store the results of the optimization.
xz_distance = np.empty_like(X, dtype=float)

# Iterate over every grid point.
for counter, (idx, jdx) in enumerate(indices, 1):
    t0 = time.time()

    # Read Bloch lengths for each point.
    x_val = X[idx, jdx]
    z_val = Z[idx, jdx]

    # Construct Boch length constraints.
    Rt = {(1,): float(x_val), (2,): 0.0, (1, 2): float(z_val)}

    # Solve the optmization problem. Find the closest quantum state with the given moments.
    param_res = compute_initial_param_repeat(d, tensor_basis, subset_index_map, Rt)
    xz_distance[idx, jdx] = sum(list(param_res.checks["moments_distance"].values()))

    tf = time.time()
    times.append(tf-t0)
    
    # Display status of the loop.
    percent_complete = (counter / total_points) * 100
    print(f"\rProgress: {percent_complete:.2f}% ({counter}/{total_points}) | Time for this point: {tf-t0:.3f} s", end="", flush=True)

print(f"\n\nDone!")
print(f"Total time: {sum(times)/60:.2f} min | Average time per point: {sum(times)/total_points:.3f} s", )

# Save

Results from simulations that have not been proccessed are stored in `data/raw/two_qutrits`.

In [ ]:
# Define paths for relevant directories.
PROJECT_ROOT = find_project_root()
data_dir = PROJECT_ROOT / "data" / "examples" / "two_qutrits"
# Create directories if they don't exist.
data_dir.mkdir(parents=True, exist_ok=True)

np.savez(data_dir / "bloch_distance.npz",
         sym=sym_distance,
         xz=xz_distance)